In [6]:
import pandas as pd
import numpy as np


In [7]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv')

In [8]:
df.columns

Index(['Listing Title', 'area_m2', 'Distances to Railway Stations',
       'Building Age', 'Stories Tall', 'Floor', 'Price',
       'Administrative Costs', 'Deposit', 'Gratuity', 'Rooms', 'Square Meters',
       'Links to Listing', 'jnc', 'Price_yen', 'Administrative Costs_yen',
       'Deposit_yen', 'Gratuity_yen', 'Admin_yen', 'deposit_months',
       'gratuity_months', 'lookalike_id', 'stories_above', 'stories_below',
       'floor_min', 'floor_max', 'is_multi_floor', 'Building_Age_Parsed',
       'age_years', 'is_new', 'age_is_lower_bound', 'is_hiraya',
       'flag_floor_above_building', 'flag_area_small', 'flag_area_large',
       'flag_fee_tiny', 'no_gratuity', 'n_rooms', 'has_L', 'has_D', 'has_K',
       'has_S', 'is_1R', 'area_per_room', 'flag_rooms_implausible',
       'flag_layout_area_conflict', 'walk_min_nearest', 'walk_min_second',
       'n_walk_stations', 'n_lines', 'walk_le_5', 'walk_le_10', 'walk_le_15',
       'has_bus_access', 'has_car_access', 'location', 'municipa

In [9]:


import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {"madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"]}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

base_maes = fold_maes(base)

with_madori_maes = fold_maes(base + groups["madori"])

diff = with_madori_maes - base_maes

print(f"Base mean MAE: {base_maes.mean():,.2f}")
print(f"With madori mean MAE: {with_madori_maes.mean():,.2f}")
print(f"Mean difference: {diff.mean():,.2f}")
print(f"Difference std: {diff.std():,.2f}")
print(f"Folds improved: {(diff < 0).sum()} / {len(diff)}")


c:\Users\priya\Desktop\data science try\tokyo-rental-intelligence\.venv\Lib\site-packages\xgboost\core.py:569: UserWarning: [13:50:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Base mean MAE: 11,302.69
With madori mean MAE: 11,179.41
Mean difference: -123.28
Difference std: 94.34
Folds improved: 5 / 5


In [11]:
df["age_ge_45"] = (df["age_years"] >= 45).astype(int)
splits = list(GroupKFold(n_splits=5).split(df, df["Price_yen"], groups=df["group_coarse"]))
base = ["area_m2", "age_years", "floor_min", "stories_above", "walk_min_nearest"]
cat = ["municipality"]
groups = {
    "is_new": ["is_new"],
    "madori": ["n_rooms","has_L","has_D","has_K","has_S","is_1R"],
    "station": ["walk_min_second","n_lines","has_bus_access"],
    "age_ge_45": ["age_ge_45"],
    "extras": ["is_multi_floor","is_hiraya","stories_below"],
}

def make_pipe(num_cols):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat)
        ])),
        ("regressor", XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=6,
                                   tree_method="hist", device="cuda", random_state=0, n_jobs=-1))
    ])

def fold_maes(num_cols):
    maes = []
    for tr, te in splits:
        pipe = make_pipe(num_cols)
        pipe.fit(df.iloc[tr][num_cols + cat], df.iloc[tr]["Price_yen"])
        preds = pipe.predict(df.iloc[te][num_cols + cat])
        maes.append(mean_absolute_error(df.iloc[te]["Price_yen"], preds))
    return np.array(maes)

results = {}
for name, cols in groups.items():
    d = fold_maes(base + cols) - base_maes
    results[name] = d
    print(f"{name:10s} diff {d.mean():8.1f} ± {d.std():6.1f}  improved {(d<0).sum()}/5")


is_new     diff     17.1 ±   33.9  improved 2/5
madori     diff   -123.3 ±   94.3  improved 5/5
station    diff    -54.3 ±  160.9  improved 3/5
age_ge_45  diff      0.0 ±    0.0  improved 0/5
extras     diff    100.6 ±  128.0  improved 2/5


In [12]:
df[["is_new", "age_ge_45"]].describe()

,is_new,age_ge_45
count,183490.000000,183490.000000
mean,0.157845,0.039986
std,0.364597,0.195926
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,0.000000,0.000000
max,1.000000,1.000000


# H1 and H4: ablate nonlinear terms on the log-area + log-target linear model, GroupKFold paired folds.
# Imports: numpy as np, pandas as pd, GroupKFold, mean_absolute_error, ColumnTransformer,
#   Pipeline, SimpleImputer, StandardScaler, OneHotEncoder, SplineTransformer,
#   FunctionTransformer, LinearRegression, TransformedTargetRegressor.
#
# Reuse the SAME splits as the XGBoost ablation: splits = list(GroupKFold(5).split(df, df["Price_yen"], groups=df["group_coarse"])).
# Create df["age_ge_45"] = (df["age_years"] >= 45).astype(int) if it does not exist.
#
# def make_linear(walk_mode, age_mode, extra_cols=[]): return a TransformedTargetRegressor(
#     regressor=Pipeline([("pre", ColumnTransformer([...])), ("lr", LinearRegression())]),
#     func=np.log1p, inverse_func=lambda z: np.clip(np.expm1(z), 1e4, 5e6))
# ColumnTransformer pieces (every learned step inside the pipeline):
#   ("area", Pipeline([("log", FunctionTransformer(np.log)), ("sc", StandardScaler())]), ["area_m2"])
#   ("fl", Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())]),
#       ["floor_min", "stories_above"])
#   ("cat", OneHotEncoder(handle_unknown="ignore"), ["municipality"])
#   walk_mode "raw":    ("walk", Pipeline([imputer median + add_indicator, StandardScaler]), ["walk_min_nearest"])
#   walk_mode "spline": ("walk", Pipeline([imputer median + add_indicator, SplineTransformer(n_knots=6, degree=3, knots="quantile")]), ["walk_min_nearest"])
#   walk_mode "flags":  ("walk", Pipeline([imputer median + add_indicator, StandardScaler]), ["walk_min_nearest"]) plus passthrough of ["walk_le_5","walk_le_10","walk_le_15"] (fill NaN with 0)
#   age_mode "raw":     age_years -> StandardScaler
#   age_mode "spline":  age_years -> SplineTransformer(n_knots=7, degree=3, knots="quantile")
#   extra_cols: passthrough columns such as ["age_ge_45", "is_new"].
#
# Runs (each returns 5 fold MAEs on raw yen, fit on train rows only, no tuning):
#   A: walk raw,    age raw                    <- base
#   B: walk spline, age raw                    (H1)
#   C: walk flags,  age raw                    (H1, threshold version)
#   D: walk raw,    age spline                 (age nonlinearity)
#   E: walk raw,    age raw,    extra ["age_ge_45"]            (H4 on a linear age term)
#   F: walk raw,    age spline, extra ["age_ge_45"]            (H4 on top of a flexible age term)
# Print for each run vs A: mean MAE, mean diff, diff std, folds improved.
# Also print MAE within the age_years >= 45 segment only for D and F (H4 is low-powered overall).
# Do not modify splits. Do not tune.

In [13]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, SplineTransformer, FunctionTransformer
from sklearn.linear_model import LinearRegression
from sklearn.compose import TransformedTargetRegressor

if "age_ge_45" not in df.columns:
    df["age_ge_45"] = (df["age_years"] >= 45).astype(int)


def make_linear(walk_mode, age_mode, extra_cols=None):
    extra_cols = extra_cols or []

    transformers = [
        ("area", Pipeline([
            ("log", FunctionTransformer(np.log)),
            ("sc", StandardScaler())
        ]), ["area_m2"]),

        ("fl", Pipeline([
            ("imp", SimpleImputer(strategy="median", add_indicator=True)),
            ("sc", StandardScaler())
        ]), ["floor_min", "stories_above"]),

        ("cat", OneHotEncoder(handle_unknown="ignore"), ["municipality"]),
    ]

    if walk_mode == "raw":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("sc", StandardScaler())
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "spline":
        transformers.append((
            "walk",
            Pipeline([
                ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                ("spline", SplineTransformer(
                    n_knots=6, degree=3, knots="quantile"
                ))
            ]),
            ["walk_min_nearest"]
        ))
    elif walk_mode == "flags":
        transformers.extend([
            (
                "walk",
                Pipeline([
                    ("imp", SimpleImputer(strategy="median", add_indicator=True)),
                    ("sc", StandardScaler())
                ]),
                ["walk_min_nearest"]
            ),
            (
                "walk_flags",
                SimpleImputer(strategy="constant", fill_value=0),
                ["walk_le_5", "walk_le_10", "walk_le_15"]
            )
        ])

    if age_mode == "raw":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler())
            ]),
            ["age_years"]
        ))
    elif age_mode == "spline":
        transformers.append((
            "age",
            Pipeline([
                ("imp", SimpleImputer(strategy="median")),
                ("spline", SplineTransformer(
                    n_knots=7, degree=3, knots="quantile"
                ))
            ]),
            ["age_years"]
        ))

    if extra_cols:
        transformers.append((
            "extra",
            SimpleImputer(strategy="constant", fill_value=0),
            extra_cols
        ))

    feature_cols = list(dict.fromkeys(
        col for _, _, cols in transformers for col in cols
    ))

    return TransformedTargetRegressor(
        regressor=Pipeline([
            ("pre", ColumnTransformer(transformers)),
            ("lr", LinearRegression())
        ]),
        func=np.log1p,
        inverse_func=lambda z: np.clip(np.expm1(z), 1e4, 5e6)
    ), feature_cols


runs = {
    "A": ("raw", "raw", []),
    "B": ("spline", "raw", []),
    "C": ("flags", "raw", []),
    "D": ("raw", "spline", []),
    "E": ("raw", "raw", ["age_ge_45"]),
    "F": ("raw", "spline", ["age_ge_45"]),
}

run_maes = {}
segment_maes = {}

for name, (walk_mode, age_mode, extra_cols) in runs.items():
    fold_maes = []
    fold_segment_maes = []

    model, feature_cols = make_linear(walk_mode, age_mode, extra_cols)

    for tr, te in splits:
        model.fit(df.iloc[tr][feature_cols], df.iloc[tr]["Price_yen"])
        preds = model.predict(df.iloc[te][feature_cols])

        y_test = df.iloc[te]["Price_yen"].to_numpy()
        fold_maes.append(mean_absolute_error(y_test, preds))

        if name in {"D", "F"}:
            mask = df.iloc[te]["age_years"].to_numpy() >= 45
            fold_segment_maes.append(
                mean_absolute_error(y_test[mask], preds[mask])
            )

    run_maes[name] = np.array(fold_maes)

    if name in {"D", "F"}:
        segment_maes[name] = np.array(fold_segment_maes)


a_maes = run_maes["A"]

for name, maes in run_maes.items():
    diff = maes - a_maes
    print(
        f"{name}: MAE {maes.mean():,.2f}, "
        f"mean diff {diff.mean():,.2f}, "
        f"diff std {diff.std():,.2f}, "
        f"improved {(diff < 0).sum()}/5"
    )

for name in ["D", "F"]:
    print(
        f"{name} age>=45 MAE: "
        f"{segment_maes[name].mean():,.2f}"
    )

A: MAE 12,924.15, mean diff 0.00, diff std 0.00, improved 0/5
B: MAE 12,897.65, mean diff -26.50, diff std 17.17, improved 5/5
C: MAE 12,909.98, mean diff -14.17, diff std 15.11, improved 3/5
D: MAE 12,765.32, mean diff -158.83, diff std 11.66, improved 5/5
E: MAE 12,863.38, mean diff -60.77, diff std 8.54, improved 5/5
F: MAE 12,765.75, mean diff -158.40, diff std 11.81, improved 5/5
D age>=45 MAE: 11,592.86
F age>=45 MAE: 11,590.71


In [16]:

FEATURES = [
    "area_m2", "age_years", "floor_min", "stories_above",
    "walk_min_nearest", "n_rooms", "has_L", "has_D", "has_K", "has_S", "is_1R"
]
CAT = ["municipality"]

oof = pd.Series(np.nan, index=df.index, dtype=float)
df["fold"] = np.nan

for fold_id, (tr, te) in enumerate(splits):
    pipe = make_pipe(FEATURES)
    train_cols = FEATURES + CAT

    pipe.fit(
        df.iloc[tr][train_cols],
        df.iloc[tr]["Price_yen"]
    )
    oof.iloc[te] = pipe.predict(df.iloc[te][train_cols])
    df.iloc[te, df.columns.get_loc("fold")] = fold_id

assert oof.isna().sum() == 0

abs_err = (oof - df["Price_yen"]).abs()
pct_err = abs_err / df["Price_yen"]

print(f"Overall MAE: {abs_err.mean():,.2f}")


def error_table(group):
    return (
        pd.DataFrame({
            "group": group,
            "abs_err": abs_err,
            "pct_err": pct_err,
        })
        .groupby("group", dropna=False)
        .agg(
            n=("abs_err", "size"),
            MAE=("abs_err", "mean"),
            median_abs_error=("abs_err", "median"),
            median_pct_error=("pct_err", "median"),
        )
        .reset_index()
    )


price_band = pd.cut(
    df["Price_yen"],
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"]
)

if "region" in df.columns:
    region = df["region"].fillna("unknown")
else:
    municipality = df["municipality"].astype("string")
    region = pd.Series(
        np.select(
            [
                municipality.str.endswith("区", na=False),
                municipality.str.endswith("市", na=False),
                municipality.str.endswith(("町", "村"), na=False),
            ],
            ["ward", "city", "town_village"],
            default="unknown",
        ),
        index=df.index,
    )

area_band = pd.cut(
    df["area_m2"],
    [0, 20, 30, 50, 80, np.inf],
    labels=["<20", "20-30", "30-50", "50-80", ">80"]
)

print("\nPrice bands")
display(error_table(price_band))

print("\nRegions")
display(error_table(region))

print("\nArea bands")
display(error_table(area_band))

output_path = "../data/interim/oof_xgb_groupkfold.csv"
oof.rename("oof_prediction").to_csv(output_path, index=True)
print(f"\nSaved OOF predictions to {output_path}")



Overall MAE: 11,179.41

Price bands


,group,n,MAE,median_abs_error,median_pct_error
0,<80k,88346,5983.820677,4778.847656,0.080524
1,80-150k,74126,10528.769978,8112.328125,0.078989
2,>150k,21018,35312.999162,20155.460938,0.098709



Regions


,group,n,MAE,median_abs_error,median_pct_error
0,city,59320,7400.592017,5201.470703,0.084243
1,town_village,317,6409.325907,3526.406250,0.076264
2,ward,123853,13001.509584,7511.039062,0.080707



Area bands


,group,n,MAE,median_abs_error,median_pct_error
0,<20,35424,5541.163806,4412.919922,0.083866
1,20-30,67608,7989.017870,6087.630859,0.077482
2,30-50,49644,10539.086331,7455.281250,0.078896
3,50-80,26009,17927.752654,11409.984375,0.092923
4,>80,4805,67723.998735,29291.250000,0.127067



Saved OOF predictions to ../data/interim/oof_xgb_groupkfold.csv


In [ ]:
FEATURES = [
    "area_m2", "age_years", "floor_min", "stories_above",
    "walk_min_nearest", "n_rooms", "has_L", "has_D",
    "has_K", "has_S", "is_1R"
]
CAT = ["municipality"]
train_cols = FEATURES + CAT

def make_q_pipe(alpha):
    return Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", "passthrough", FEATURES),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CAT),
        ])),
        ("regressor", XGBRegressor(
            objective="reg:quantileerror",
            quantile_alpha=alpha,
            n_estimators=400,
            learning_rate=0.05,
            max_depth=6,
            tree_method="hist",
            device="cuda",
            random_state=0,
            n_jobs=-1,
        )),
    ])

intervals = pd.DataFrame(
    np.nan,
    index=df.index,
    columns=["lo", "mid", "hi"],
)
intervals["fold"] = np.nan

z = np.log1p(df["Price_yen"])

for fold_id, (tr, te) in enumerate(splits):
    X_train = df.iloc[tr][train_cols]
    X_test = df.iloc[te][train_cols]
    y_train = z.iloc[tr]

    for alpha, column in [(0.1, "lo"), (0.5, "mid"), (0.9, "hi")]:
        q_pipe = make_q_pipe(alpha)
        q_pipe.fit(X_train, y_train)

        predictions = np.expm1(q_pipe.predict(X_test))
        intervals.loc[df.iloc[te].index, column] = predictions

    intervals.loc[df.iloc[te].index, "fold"] = fold_id

assert intervals[["lo", "mid", "hi"]].notna().all().all()

intervals["lo"] = np.minimum(intervals["lo"], intervals["mid"])
intervals["hi"] = np.maximum(intervals["hi"], intervals["mid"])

y = df["Price_yen"]
inside = y.between(intervals["lo"], intervals["hi"])
width_pct = (intervals["hi"] - intervals["lo"]) / intervals["mid"]

print(f"Coverage: {inside.mean():.4f} (target 0.8000)")
print(f"Mean width (%): {width_pct.mean():.4f}")
print(f"Median width (%): {width_pct.median():.4f}")
print(f"Median MAE: {(intervals['mid'] - y).abs().mean():,.2f}")


def interval_summary(group):
    return (
        pd.DataFrame({
            "group": group,
            "inside": inside,
            "width_pct": width_pct,
        })
        .groupby("group", dropna=False)
        .agg(
            coverage=("inside", "mean"),
            median_width_pct=("width_pct", "median"),
            n=("inside", "size"),
        )
        .reset_index()
    )


price_band = pd.cut(
    y,
    [0, 80_000, 150_000, np.inf],
    labels=["<80k", "80-150k", ">150k"],
)

area_band = pd.cut(
    df["area_m2"],
    [0, 20, 30, 50, 80, np.inf],
    labels=["<20", "20-30", "30-50", "50-80", ">80"],
)

print("\nPrice bands")
display(interval_summary(price_band))

print("\nArea bands")
display(interval_summary(area_band))

print("\nRegions")
display(interval_summary(df["region"].fillna("unknown")))

print("\nFolds")
display(interval_summary(intervals["fold"].astype("Int64")))

interval_output_path = "../data/interim/quantile_intervals_groupkfold.csv"
intervals[["lo", "mid", "hi"]].to_csv(interval_output_path, index=True)

print(f"\nSaved intervals to {interval_output_path}")